# ДЗ Модуль 02 — Vector Search

LLM Zoomcamp 2026, DataTalksClub.

Эмбеддинги считаем лёгким ONNX-рантаймом (без PyTorch/CUDA), база знаний — 72 страницы уроков курса с коммита `8c1834d`.

**Подготовка окружения:**
```bash
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
python download.py   # один раз — скачать ONNX-модель в models/
```
Рядом с ноутбуком должны лежать helper-скрипты `download.py` и `embedder.py` из `02-vector-search/embed/`.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

embed = Embedder()

## Q1. Embedding a query

Эмбеддим запрос и смотрим первое значение вектора `v[0]`.

In [ ]:
v = embed.encode("How does approximate nearest neighbor search work?")
print(f"len(v) = {len(v)}")
print(f"v[0] = {v[0]:.4f}")

len(v) = 384
v[0] = -0.0206


**Ответ Q1:** `v[0] = -0.0206` → **-0.02**

## Loading the data

Тянем страницы уроков с фиксированного коммита `8c1834d` — те же 72 страницы.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"{len(documents)} pages")

72 pages


## Q2. Cosine similarity

Вектора нормированы, поэтому dot-product = косинусная близость. Берём страницу `07-sqlitesearch-vector.md`, эмбеддим её `content` и считаем близость с вектором запроса из Q1.

In [ ]:
target_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_name)

doc_vec = embed.encode(target_doc["content"])
cos = float(doc_vec.dot(v))
print(f"cosine similarity = {cos:.4f}")

cosine similarity = 0.3611


**Ответ Q2:** `0.3611` → **0.37**

## Q3. Chunking and search by hand

Режем страницы на перекрывающиеся чанки, эмбеддим каждый чанк через `encode_batch`, складываем в матрицу `X` и считаем `scores = X.dot(v)`.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"chunks: {len(chunks)}")

chunk_texts = [c["content"] for c in chunks]
batch_size = 50
X = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    X.extend(embed.encode_batch(chunk_texts[i:i + batch_size]))
X = np.array(X)

scores = X.dot(v)
best = int(np.argmax(scores))
print(f"top chunk filename = {chunks[best]['filename']} (score={scores[best]:.4f})")

chunks: 295
top chunk filename = 02-vector-search/lessons/07-sqlitesearch-vector.md (score=0.6489)


**Ответ Q3:** **`02-vector-search/lessons/07-sqlitesearch-vector.md`**

## Q4. Vector search with minsearch

Индексируем чанки через `VectorSearch` и ищем по новому запросу.

In [ ]:
vindex = VectorSearch()
vindex.fit(X, chunks)

v_q4 = embed.encode("What metric do we use to evaluate a search engine?")
vres = vindex.search(v_q4, num_results=5)
print(f"first result filename = {vres[0]['filename']}")

first result filename = 04-evaluation/lessons/05-search-metrics.md


**Ответ Q4:** **`04-evaluation/lessons/05-search-metrics.md`**

## Q5. Text search vs vector search

Те же чанки индексируем текстовым `Index` (поле `content`). Запускаем оба поиска, берём топ-5 каждого и ищем файл, который есть в векторных результатах, но отсутствует в текстовых.

In [ ]:
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embed.encode(query_q5)

vector_results_q5 = vindex.search(v_q5, num_results=5)
text_results_q5 = tindex.search(query_q5, num_results=5)

vector_files = {r["filename"] for r in vector_results_q5}
text_files = {r["filename"] for r in text_results_q5}
print(f"in vector but not text = {sorted(vector_files - text_files)}")

in vector but not text = ['02-vector-search/lessons/08-pgvector.md']


**Ответ Q5:** **`02-vector-search/lessons/08-pgvector.md`**

## Q6. Hybrid search (RRF)

Объединяем векторный и текстовый поиск через Reciprocal Rank Fusion: каждый документ получает `1 / (k + rank)` из каждого списка, где он встречается (`k = 60`).

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query_q6 = "How do I give the model access to tools?"
v_q6 = embed.encode(query_q6)

vector_results = vindex.search(v_q6, num_results=5)
text_results = tindex.search(query_q6, num_results=5)
fused = rrf([vector_results, text_results])
print(f"RRF first result filename = {fused[0]['filename']}")

RRF first result filename = 01-agentic-rag/lessons/13-function-calling.md


**Ответ Q6:** **`01-agentic-rag/lessons/13-function-calling.md`** (не первый ни в одном поиске по отдельности — выигрывает за счёт высокого ранга в обоих).

## Итоговые ответы

| Вопрос | Ответ |
|--------|-------|
| Q1 | -0.02 |
| Q2 | 0.37 |
| Q3 | `02-vector-search/lessons/07-sqlitesearch-vector.md` |
| Q4 | `04-evaluation/lessons/05-search-metrics.md` |
| Q5 | `02-vector-search/lessons/08-pgvector.md` |
| Q6 | `01-agentic-rag/lessons/13-function-calling.md` |